In [87]:
import pandas as pd
#!pip3 install pickle5
import pickle5 as pickle
import os
def load_pickle(pickle_path):
    with open(pickle_path, 'rb') as f:
        pkl_file = pickle.load(f)
    return pkl_file

%cd /home/jupyter/MRI-spleen/predictCAD

/home/jupyter/MRI-spleen/predictCAD


In [80]:
cohort_name = 'cohorts/CADcohort_all_rad.csv'
feats_of_interest = ['spleen_original_shape_Sphericity',
                     'spleen_original_shape_MinorAxisLength',
                     'spleen_original_gldm_DependenceEntropy']
seg_fname = '2_wat.nii.gz' 
stitch_fname = 'wat.nii.gz'

In [81]:
coh = pd.read_csv(cohort_name)
print('ID' in coh.columns)
coh.head()

True


,age,tchol,hdl,SBP,dm2_prev,dm1_prev,antihtnbase,statin0,spleen_original_shape_Elongation,spleen_original_shape_Flatness,...,race_black,race_mixed,race_other,race_white,sex_Female,sex_Male,smoking_Current,smoking_Never,smoking_Previous,train
0,49,199.845320,61.059551,153.0,0,0,0,0,0.543486,0.402501,...,0,0,0,1,1,0,0,0,1,1
1,59,239.211135,55.220417,130.0,0,0,0,0,0.516462,0.380680,...,0,0,0,1,1,0,0,0,1,1
2,64,268.174786,72.351121,153.0,0,0,0,0,0.653198,0.395604,...,0,0,0,1,1,0,0,1,0,1
3,56,242.459395,54.911059,113.0,0,0,0,0,0.643589,0.404867,...,0,0,0,1,1,0,0,1,0,1
4,61,221.848413,45.978345,145.0,0,0,0,0,0.674545,0.444059,...,0,0,0,1,1,0,0,1,0,1


In [73]:
feat_to_lows = {}
feat_to_highs = {}
for feat in feats_of_interest:
    coh = coh.sort_values(by=[feat], ascending = True)
    feat_to_lows[feat] = list(coh.ID)[:3]
    feat_to_highs[feat] = list(coh.ID)[-3:]
    

In [74]:
stitched_dir = '/home/jupyter/MRI-spleen/stitched_data'
nnunet_folder = '/home/jupyter/MRI-spleen/nnunet_data'

orig_conversion_map = load_pickle(os.path.join(nnunet_folder, 'conversion.pkl'))
conversion_map = {}
for entry in orig_conversion_map:
    entry = entry[0]    
    for img_path in entry['img_paths']:
        conversion_map[img_path['orig']] = img_path['nnunet'] 


In [16]:
print(feat_to_lows)
print(feat_to_highs)

{'spleen_original_shape_Sphericity': [3274724, 2127470, 4686499], 'spleen_original_shape_MinorAxisLength': [3698382, 4107768, 1179278], 'spleen_original_gldm_DependenceEntropy': [2984281, 4107768, 1276646]}
{'spleen_original_shape_Sphericity': [2348684, 2984281, 1468928], 'spleen_original_shape_MinorAxisLength': [2587355, 3684691, 4184979], 'spleen_original_gldm_DependenceEntropy': [5996452, 1554167, 5399282]}


In [82]:
%mkdir images
%cd images/

mkdir: cannot create directory ‘images’: File exists
/home/jupyter/MRI-spleen/predictCAD/images


In [83]:
for feat in feat_to_lows:
    fold_name = feat.replace('spleen_original_', '')
    #%mkdir "$fold_name"
    %cd "$fold_name"
    for ID in feat_to_lows[feat]:
        from_seg_fname = '../../../segments/'+str(ID)+'/' + seg_fname
        to_seg_fname = str(ID)+'_low_'+seg_fname
        %cp "$from_seg_fname" "$to_seg_fname"
        
        stitched_ext = stitched_dir+'/'+str(ID)+'/'+stitch_fname
        from_stitched = conversion_map[stitched_ext]
        print(from_stitched)
        to_stitched = str(ID)+'_low_'+stitch_fname
        %cp "$from_stitched" "$to_stitched"
        break
        
    for ID in feat_to_highs[feat]:
        from_seg_fname = '../../../segments/'+str(ID)+'/' + seg_fname
        to_seg_fname = str(ID)+'_high_'+seg_fname
        %cp "$from_seg_fname" "$to_seg_fname"
        
        stitched_ext = stitched_dir+'/'+str(ID)+'/'+stitch_fname
        from_stitched = conversion_map[stitched_ext]
        to_stitched = str(ID)+'_high_'+stitch_fname
        %cp "$from_stitched" "$to_stitched"
        break

    %cd ..

        

/home/jupyter/MRI-spleen/predictCAD/images/shape_Sphericity
cp: error writing '3274724_low_2_wat.nii.gz': No space left on device
/home/jupyter/MRI-spleen/nnunet_data/ukbb_4ch_19747_0000.nii.gz
cp: error writing '3274724_low_wat.nii.gz': No space left on device
cp: error writing '2348684_high_2_wat.nii.gz': No space left on device
cp: error writing '2348684_high_wat.nii.gz': No space left on device
/home/jupyter/MRI-spleen/predictCAD/images
/home/jupyter/MRI-spleen/predictCAD/images/shape_MinorAxisLength
cp: error writing '3698382_low_2_wat.nii.gz': No space left on device
/home/jupyter/MRI-spleen/nnunet_data/ukbb_4ch_23348_0000.nii.gz
cp: error writing '3698382_low_wat.nii.gz': No space left on device
cp: error writing '2587355_high_2_wat.nii.gz': No space left on device
cp: error writing '2587355_high_wat.nii.gz': No space left on device
/home/jupyter/MRI-spleen/predictCAD/images
/home/jupyter/MRI-spleen/predictCAD/images/gldm_DependenceEntropy
cp: error writing '2984281_low_2_wat.ni

In [84]:
'''
Compile from logfile.log
'''

'\nCompile from logfile.log\n'

In [93]:
f = open('logfile-Copy1.log', 'w+')

In [96]:
with open('logfile-Copy1.log') as f:
    f = f.readlines()

auc_inds = []
for ind, elem in enumerate(f):
    if 'AUC' in elem:
        auc_inds.append(ind)
auc_inds

[]